# 06 · Making the application trustworthy

**Session 01 · notebook 6 of 6**

## Objective

Add verification of every number in an answer, with the program's `pcn-core` package, and an evaluation suite that runs the application over a set of questions with known evidence. Compare two configurations of the application on the same suite.

## The checks

| Check | Question it answers | Where it runs |
|---|---|---|
| Retrieval | Did the application find the right evidence? | Notebook 03, section 6 |
| Grounding | Is the answer written from that evidence? | Citations in the answer |
| Verification | Does each number match its source record? | Claim tags and `pcn-core`, this notebook |
| Evaluation | Does the application behave across many questions? | The suite, this notebook |

## The application after this notebook

Diagram: the question goes to an agent with tools, which produces an answer with claim tags; pcn-core marks each number as verified, flagged or unmarked. Separately, the evaluation suite produces a table per question.

The verifier and the evaluation suite are what this notebook adds.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # the program's repository
BRANCH = "feat/training-series"   # change to "main" once the series is merged
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


In [2]:
import json
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

docs = w.load_documents()
store = w.build_store(w.split_sections(docs))
search_documents = w.make_search_tool(store, k=3, since="2020")
tools = [search_documents, w.get_indicator]

INSTRUCTIONS_CLAIMS = """You are a research assistant for development data.
Use get_indicator for every statistic and search_documents for definitions and context. Do not quote figures from memory.
Write every number that comes from get_indicator inside a claim tag that names its claim_id, like
<claim id="IT.NET.USER.ZS:KEN:2024" policy="rounded" decimals="1">35.0</claim>.
Name the indicator and the source. If the tools do not provide what is needed, say so.
If the question is ambiguous, ask which reading is meant."""

## 1. Claim tags

The instructions ask the model to wrap each figure in a tag that names the observation it came from. `w.verify` finds the tags and hands each number and its record to `pcn_core.compare_by_policy` with the policy from the tag: rounded to the stated decimals, or exact. This is the Proof-Carrying Numbers protocol used in the program's chatbot, with the same comparison code.

In [3]:
results = [w.indicator_lookup(i, c, start=2010) for i in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"] for c in ["Kenya", "Rwanda"]]
observations = w.observations_from_results(results)
print(len(observations), "observations available to the verifier, for example", list(observations)[:2])

w.use("nb06_claims_main")
reply = model.invoke([SystemMessage(content=INSTRUCTIONS_CLAIMS), HumanMessage(content="DATA:\n" + json.dumps(results) + "\n\nQUESTION: " + w.MAIN_QUESTION)])
print("\nAs the reader sees it:\n")
w.show(w.strip_claims(reply.content))
print("\nVerification:\n")
w.print_checks(w.verify(reply.content, observations))

60 observations available to the verifier, for example ['IT.NET.USER.ZS:KEN:2010', 'IT.NET.USER.ZS:KEN:2011']

As the reader sees it:

Internet use reached 35.0 percent of the population in Kenya and 31.7 percent in Rwanda in 2024, up
from 7.2 and 8.0 percent in 2010 (World Development Indicators, IT.NET.USER.ZS). GDP per capita in
current US dollars was 2,133 in Kenya and 1,060 in Rwanda in 2024, up from 1,092 and 587 in 2010
(NY.GDP.PCAP.CD). Kenya has the higher level on both indicators and Rwanda the faster growth on
both; internet use in the two countries converged to about a third of the population.

Verification:

✓ verified        35.0  IT.NET.USER.ZS:KEN:2024      record holds 34.9762
✓ verified        31.7  IT.NET.USER.ZS:RWA:2024      record holds 31.7033
✓ verified         7.2  IT.NET.USER.ZS:KEN:2010      record holds 7.2
✓ verified         8.0  IT.NET.USER.ZS:RWA:2010      record holds 8
✓ verified       2,133  NY.GDP.PCAP.CD:KEN:2024      record holds 2133.46
✓ verified 

Every tagged number matches its record. Years are not claims and are skipped.

## 2. A wrong number and a number without a claim

The planted problem. One tagged figure has been altered, and one figure has no tag because it came from nowhere in the data. The verifier flags the first and marks the second as unmarked: it carries no claim, so it receives no mark, whether or not it happens to be right.

In [4]:
w.use("nb06_claims_error")
reply = model.invoke([SystemMessage(content=INSTRUCTIONS_CLAIMS), HumanMessage(content="DATA:\n" + json.dumps(results) + "\n\nQUESTION: " + w.MAIN_QUESTION)])
w.show(w.strip_claims(reply.content))
print()
w.print_checks(w.verify(reply.content, observations))

Internet use reached 35.0 percent of the population in Kenya and 35.7 percent in Rwanda in 2024
(World Development Indicators, IT.NET.USER.ZS). GDP per capita in current US dollars was 2,133 in
Kenya and 1,060 in Rwanda. About 70 percent of adults in both countries use mobile money, which runs
on basic phones and does not require internet access.

✓ verified        35.0  IT.NET.USER.ZS:KEN:2024      record holds 34.9762
✗ flagged         35.7  IT.NET.USER.ZS:RWA:2024      record holds 31.7033; shown 35.7
✓ verified       2,133  NY.GDP.PCAP.CD:KEN:2024      record holds 2133.46
✓ verified       1,060  NY.GDP.PCAP.CD:RWA:2024      record holds 1059.94
· unmarked          70                               no claim tag, so no mark is drawn

3 verified, 1 flagged, 1 unmarked


A reader of the chatbot sees a mark beside each number. A flagged number is one the model changed. An unmarked number is one the model did not tie to a record, and the fail-closed rule says it gets no mark.

What the check says nothing about: whether the record is the right one for the question, whether the method behind it is sound, or whether the sentence around the number interprets it correctly. Those stay with the reader.

## 3. Two configurations of the application

- **A, retrieval only**: the RAG chain of notebook 03. Passages as context, no tools, no tags.
- **B, tools and claims**: the agent of notebook 05 with the claim instructions, and the verifier.

In [5]:
INSTRUCTIONS_RAG = """You are a research assistant for development data.
Answer only from the supplied passages and cite them in the square-bracket form given. If they do not contain what is needed, say so."""
rag_prompt = ChatPromptTemplate.from_messages([("system", INSTRUCTIONS_RAG), ("human", "PASSAGES:\n{context}\n\nQUESTION: {question}")])
rag_chain = rag_prompt | model
agent = create_agent(model, tools, system_prompt=INSTRUCTIONS_CLAIMS)


def app_rag(question, scenario):
    hits = store.similarity_search(question, k=4)
    w.use(scenario)
    reply = rag_chain.invoke({"context": w.format_context(hits), "question": question})
    return reply.content, hits, {}


def app_tools(question, scenario):
    w.use(scenario)
    result = agent.invoke({"messages": [HumanMessage(content=question)]}, config={"recursion_limit": 10})
    obs = w.observations_from_results(w.tool_results(result, "get_indicator"))
    hits = store.similarity_search(question, k=4)  # for the retrieval column; the agent may or may not have searched
    return w.final_text(result), hits, obs

## 4. The evaluation suite

Eight questions with expected documents, expected numbers, and expected behaviour, from `assets/questions.json`. For each question and configuration the table reports:

- **retrieval**: did the top-4 passages include an expected document
- **numbers**: how many expected values appear in the answer
- **cited**: does the answer cite a retrieved document or name a source
- **verify**: counts of verified, flagged, and unmarked numbers (configuration B only)

In [6]:
questions = w.load_questions()


def evaluate(app, prefix):
    rows = []
    for qd in questions:
        answer, hits, obs = app(qd["question"], f"{prefix}_{qd['id']}")
        found, total = w.numbers_present(answer, qd["expected_numbers"])
        cited = w.grounded(answer, hits) or "WDI" in answer or "World Development Indicators" in answer
        if obs:
            checks = w.verify(answer, obs)
            v = sum(c.status == "verified" for c in checks); f = sum(c.status == "flagged" for c in checks); u = sum(c.status == "unmarked" for c in checks)
            ver = f"{v}v {f}f {u}u"
        else:
            ver = "n/a" if prefix.endswith("rag") else "no data used"
        rows.append(w.EvalRow(qd["id"], qd["category"], "hit" if w.retrieval_hit(hits, qd["expected_documents"]) else "miss", f"{found}/{total}", "yes" if cited else "no", ver))
    return rows


print("A · retrieval only\n")
w.print_table(evaluate(app_rag, "nb06_rag"))
print("\n\nB · tools and claims\n")
w.print_table(evaluate(app_tools, "nb06_tools"))

A · retrieval only

id   category           retrieval  numbers  cited  verify           note
------------------------------------------------------------------------
q1   main               hit        4/4      yes    n/a              
q2   factual            hit        1/1      yes    n/a              
q3   ambiguous          hit        0/0      yes    n/a              
q4   missing-data       hit        0/0      yes    n/a              
q5   out-of-scope       hit        0/0      yes    n/a              
q6   factual            hit        2/2      yes    n/a              
q7   definition         hit        0/0      yes    n/a              
q8   outdated-evidence  hit        0/2      yes    n/a              


B · tools and claims

id   category           retrieval  numbers  cited  verify           note
------------------------------------------------------------------------
q1   main               hit        4/4      yes    4v 0f 0u         
q2   factual            hit        1/1     

## 5. Reading the table

- **q8, outdated evidence.** Configuration A answers from the 2016 summary when it ranks first and reports 2015 figures; B fetches the series and reports 2024. The expected numbers are found in B and missed in A.
- **q3, ambiguous.** Neither configuration should produce a number. B asks which reading is meant, as its instructions say; A picks one. The table shows 0/0 for both; the behaviour has to be read.
- **q4, missing data.** Both say that 2025 is unavailable; B's statement rests on an empty tool result.
- **q5, out of scope.** Both give no number. A's retrieval "hit" is the coffee note, which is the expected document and contains no figures.
- **verify.** In B every number from the data tool is verified; in A there is nothing to verify against, because no number came from a tool.

A table like this is what distinguishes a demonstration from a measured application. It is rerun after every change to the instructions, the tools, the store, or the model. For suites at scale, with a model as a judge for the behaviours a rule cannot check, see the program's evaluation cookbook; libraries such as Ragas and DeepEval provide those judges.

## 6. What the suite does not cover

- Whether the application chose the right indicator for the question. A verified number from the wrong series is verified.
- Whether the interpretation around the numbers is right.
- Behaviour on questions nobody wrote down. The suite grows from use: every failure found in use becomes a question.

## Key points

- Verification checks each number against its source record with `pcn-core` and fails closed: no claim, no mark.
- Evaluation runs the application over questions with known evidence and expected behaviour, and is rerun on every change.
- The two configurations differ most on outdated evidence and on numbers: tools make numbers checkable; retrieval alone does not.

## Exercise

Add a ninth question to `assets/questions.json` with an expected number that is deliberately wrong, rerun the suite, and confirm that the numbers column catches it. Then correct it.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.